In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from astropy import units as u

In [2]:
def toa_calulator(df):
    '''
    Return the toa of a burst
    in the case of multiple components of a single burst, take the middle of the burst
    '''
    if df.shape[0] == 1:
        mjd = df.iloc[0]["toa_bary_tdb_inf_freq"]

    elif df.shape[0] > 1:
        mjd_list = df["toa_bary_tdb_inf_freq"].values
        mjd = (mjd_list[0] + mjd_list[-1]) / 2
 
    return mjd

In [3]:
def peak_sn_calc(df):
    '''
    input: pandas dataframe where each line has the burst properties per component
    Return the maximum value of the snr of all comp
    '''
    
    num_comp = df.shape[0]
    if num_comp == 1:
        peak_sn = df.iloc[0]["peak_snr"]
    elif num_comp > 1:
        peak_list = df["peak_snr"].values
        peak_sn = np.max(peak_list)
    
    return peak_sn,num_comp

In [4]:
def bandwidth_calc(df, telescope):
    '''
    input: pandas dataframe where each line has the burst properties per component
    return the bandwith of a burst
    '''
    
    obs_band = df.f0_MHz.values[0]
    
    #Pband
    if obs_band == 328.0:
        bw_obs = 54
        
    #Lband
    elif obs_band:
        bw_obs = 128

    if telescope == 'st':
        bw_obs = 98
        
    #We take the absolute value, since the fwidth values can be negative
    #The bandwidth comes from autocorrelation and we define everything larger than FWHM as the BW
    signal_bw = int(abs(df.fwidth_sigma_MHz.values.max()) * 2.355)
    if signal_bw > bw_obs * 0.75:
        signal_bw = bw_obs

    
    return signal_bw, obs_band     

In [5]:
def isotropic_energies(fluences, distance=362.4):
    "Convert list of fluences to spectral densities"
    "Method based on fluence.py by K.Nimmo"

    #convert from jyms to jys
    fluence_jys = np.array(fluences) * 1e-3

    dis = 1.0 * u.megaparsec
    megap_cm = dis.to(u.cm).value

    #mpc to cm
    distance_lum_cm = megap_cm*distance

    #redshift correction, see evernote for details
    #redshift is from: Ravi et al, DSA-110
    #distance is also from Ravi et al, FAST converts themselves based on PLANCK 2016
    #link: https://ui.adsabs.harvard.edu/abs/2023ApJ...949L...3R/abstract
    z = 0.0771
    red_cor = 1 / (1 + z)**2

    energy_iso = fluence_jys * 4*np.pi*(distance_lum_cm**2) * 1e-23 * red_cor
    #print(energy_iso)

    return energy_iso

In [6]:
# def isotropic_energies(fluences, distance=362.4):
#     "Convert list of fluences to spectral densities"

#     #convert from jyms to jys
#     fluence_jys = np.array(fluences) * 1e-3

#     dis = 1.0 * u.megaparsec
#     megap_cm = dis.to(u.cm).value

#     #mpc to cm
#     distance_lum_cm = megap_cm*distance

#     #redshift correction, see evernote for details
#     #redshift is from: Ravi et al, DSA-110
#     #distance is also from Ravi et al, FAST converts themselves based on PLANCK 2016
#     #link: https://ui.adsabs.harvard.edu/abs/2023ApJ...949L...3R/abstract
#     z = 0.0771
#     red_cor = 1 / (1 + z)**2

#     energy_iso = fluence_jys * 4*np.pi*(distance_lum_cm**2) * 1e-23 * red_cor
#     #print(energy_iso)

#     return energy_iso

In [7]:
# def speclum(fluence, ontime, distance=362.4, z=0.0771):
#     '''
#     fluence assumed in Jy~ms
#     ontime assumed in ms
#     distance assumed in Mpc
#     '''
#     # actual ontime or total width? Width gives a lover limit on the luminosity.
    
#     #convert Jy ms to J s; and ms to s
#     fluence_jys = fluence*1e-3
#     ontime /= 1e3
    
#     #convert Mpc to cm
#     distance_lum_cm = 3.086e24*distance
#     energy_iso= fluence_jys*4*np.pi*(distance_lum_cm**2)*1e-23 / (1+z)**(2)
#     lum_spec = energy_iso/ontime
    
#     return lum_spec

In [8]:
def fluence_looper(df_fluence):
    """
    function to sum over the fluences components of bursts
    
    Input: The burst_csv file with all information, spc/sfxc/scale
    This is needed because we return a df with fluence (spc) and toa (sfxc)
    
    return: pandas dataframe with: id/fluence/telescope/c_freq/toa/width_ms
    """

    #the burst names to loop over
    exps = df_fluence['id'].unique()

    
    #Create a list with the telescope names for easy filtering later on
    exps_tel = [i.split('-')[1] for i in exps]

    #loop over every burst to sum the fluence
    info_list = []
    for exp, telescope in zip(exps, exps_tel):
        
        #print(exp)
        #Fluence: Making a small temp df for each exp, only taking the spc values
        if telescope != 'st':
            fluence_df_exp_temp = df_fluence[(df_fluence['id'] == exp) & (df_fluence['src'] == 'spc')]
            
            #TOA
            fluence_df_exp_temp_mjd = df_fluence[(df_fluence['id'] == exp) & (df_fluence['src'] == 'sfxc')]

            if exp == 'B08-wb' or exp == "B09-wb":
                toa = None
                fluence_df_exp_temp = fluence_df_exp_temp_mjd
            else:
                toa = toa_calulator(fluence_df_exp_temp_mjd) 
                
        elif telescope == 'st':
            fluence_df_exp_temp = df_fluence[df_fluence['id'] == exp]
            
            #TOA
            toa = toa_calulator(fluence_df_exp_temp) 
            
        #Calculate the total fluence
        fluence = sum(fluence_df_exp_temp['fluence_jyms'].values)
        
        #Calculate isotropic energies
        #energies_isotropic = isotropic_energies(fluence)
        
        #Calculate the max SN of a burst
        peak_sn, num_comp = peak_sn_calc(fluence_df_exp_temp)

        #Calculate the bandwith
        signal_bw, cent_freq = bandwidth_calc(fluence_df_exp_temp, telescope)
        
        #width in ms
        width_t = fluence_df_exp_temp.end_acf_range.values.max() - fluence_df_exp_temp.begin_acf_range.values.min()

        #Calculate the spectral luminosity
        #spel_lum = speclum(fluence, width_t, distance=362.4, z=0.0771)
        
        #Since the F0 is the same for all components, take all components, take the first entry
        cent_freq = fluence_df_exp_temp['f0_MHz'].values[0]
                            
        #Add to the list
        info_list.append([toa, peak_sn, fluence, num_comp, width_t,\
                          signal_bw, cent_freq])
        
    #Convert the fluence list to array
    info_list_arr = np.asarray(info_list)

    #Convert the info to dataframe
    new_data = {'id': exps, 'station': exps_tel, 'toa': info_list_arr[:,0], \
                'peak_sn': info_list_arr[:,1], 'fluence': info_list_arr[:,2], \
                'Number of components': info_list_arr[:,3],\
                'width_ms': info_list_arr[:,4], \
                'bandwidth': info_list_arr[:,5], \
                'central frequency': info_list_arr[:,6]}
    
    df_new_fluence = pd.DataFrame(data=new_data)

    #add only the index number as a seperate column for potential later use
    #df_new_fluence['burst-index'] = df_new_fluence['id'].str.split('-').str[0].str[1:].astype(int)
    
    return df_new_fluence

In [9]:
def df_to_small_table(file):
    """
    Function to load in the burst.csv file and return the shortend version of the .csv file
    """

    #load in the main csv file
    fluence_df = pd.read_csv(file, index_col=0)
    fluence_df.sort_values(by=['id'])

    # Specifically remove B08-wb; since we will add the information manually due it being from 2bit data
    #mask = fluence_df['id'].isin(['B08-wb', 'B09-wb'])
    #fluence_df = fluence_df[~mask]
    
    df_combined = fluence_looper(fluence_df)    
    df_combined.sort_values(by=['id'])
                          
    #Resort and reset the df
    #df_combined = df_combined.sort_values(by=['id'])
    #df_combined = df_combined.reset_index(drop=True)
    
    #Setting the formatting of the numbers correct
    #df_combined["Number of components"] = df_combined["Number of components"].astype(int)
    df_combined["bandwidth"] = df_combined["bandwidth"].astype(int)
    df_combined["central frequency"] = df_combined["central frequency"].astype(int)
    
    #df_combined['fluence_err'] = df_combined['fluence'] * 0.2
    df_combined['fluence_err'] = pd.to_numeric(df_combined['fluence'] * 0.2, errors='coerce').round(2)
    #df_combined['fluence_err'] = df_combined['fluence_err'].round(2)
    
    #Minimum spectral energy is on the order of 1e29 // calculated with the .min()
    # df_combined['spectral density'] /= 1e29
    # df_combined['spec_den_err'] = df_combined['spectral density'] * 0.2
    # df_combined['pectral density'] = df_combined['spectral density'].round(2)
    # df_combined['spec_den_err'] = df_combined['spec_den_err'].round(2)
    
    #Minimum spectral lum is on the order of 1e32 // calculated with the .min()
    # df_combined['spectral luminosity'] /= 1e32
    # df_combined['spec_lum_err'] = df_combined['spectral luminosity'] * 0.2
    # df_combined['spectral luminosity'] = df_combined['spectral luminosity'].round(2)
    # df_combined['spec_lum_err'] = df_combined['spec_lum_err'].round(2)

    df_combined['peak_sn'] = pd.to_numeric(df_combined['peak_sn'], errors='coerce').round(2)
    df_combined['width_ms'] = pd.to_numeric(df_combined['width_ms'], errors='coerce').round(2)
    df_combined['fluence'] = pd.to_numeric(df_combined['fluence'], errors='coerce').round(2)
    
    # df_paper = df_combined[['id', 'station', 'toa', 'peak_sn', 'fluence', 'fluence_err', "Number of components",\
    #                         'width_ms', 'spectral density', 'spec_den_err', 'spectral luminosity', 'spec_lum_err',\
    #                         'bandwidth', 'central frequency']]
    
    df_paper = df_combined[['id', 'station', 'toa', 'peak_sn', 'fluence', 'fluence_err',\
                            'width_ms', 'bandwidth']]

    ## Manually adding the TOA of B08 and B09; since we measured that indivually using Mark' code
    df_paper.loc[df_paper['id'] == 'B08-wb', 'toa'] = 60557.8651932359207422
    df_paper.loc[df_paper['id'] == 'B09-wb', 'toa'] = 60557.9123202934861183

    #Adding the Dwingeloo burst by hand
    new_row = pd.DataFrame([{
    'id': 'B06-dw',
    'station': 'dw',
    'toa': 60537.8884462584246648, 
    'peak_sn': np.NaN,
    'fluence': np.NaN,
    'fluence_err': np.NaN,
    'width_ms': 1,
    'bandwidth' : 200
    }])

    df_paper = pd.concat([df_paper, new_row], ignore_index=True)

    df_paper = df_paper.sort_values(by=['toa']).reset_index(drop=True)
    name_csv = "FRB20240619D_HyperFlash_paper_table.csv"
    #df_paper.to_csv(name_csv, header=True, index=True, na_rep='NA')
    display(df_paper)

    return 

df_to_small_table("hyperflash_mkt_burst_db.csv")
#display(fluence_df)

,id,station,toa,peak_sn,fluence,fluence_err,width_ms,bandwidth
0,B01-wb,wb,60511.994427,750.47,771.73,154.35,27.71,128
1,B02-st,st,60526.896118,17.98,100.75,20.15,6.99,98
2,B02-wb,wb,60526.896127,15.96,154.56,30.91,8.58,128
3,B03-st,st,60530.882722,11.85,70.26,14.05,24.47,98
4,B04-st,st,60535.945584,31.62,171.98,34.40,27.96,98
5,B05-st,st,60536.884172,6.06,30.58,6.12,10.49,98
6,B05-wb,wb,60536.884182,10.10,102.92,20.58,25.47,128
7,B06-dw,dw,60537.888446,NaN,NaN,NaN,1.00,200
8,B06-wb,wb,60537.888455,86.19,104.17,20.83,9.79,128
9,B07-st,st,60553.89809,10.67,27.11,5.42,6.12,98


In [10]:
def max_values():
    
    # full_db = pd.read_csv('burst_stats_r117.csv', sep=",", header=0, index_col=0, na_values='NA', engine='python')
    # idx_max_jy = full_db['peak_flux_jy'].idxmax()
    # Filter_df = full_db[full_db.index.isin([idx_max_jy])]
    # display(Filter_df)

    table_db = pd.read_csv('FRB20240619D_HyperFlash_table.csv', sep=",", header=0, index_col=0, na_values='NA', engine='python')

    # idx_max_jy = table_db['fluence'].idxmax()
    # Filter_df = table_db[table_db.index.isin([idx_max_jy])]
    # display(Filter_df)

    # db_l = table_db[table_db['bandwidth'] != 54]
    # idx_max_jy = db_l['fluence'].idxmax()
    # Filter_df = db_l[db_l.index.isin([idx_max_jy])]
    # display(Filter_df)

    db_l = table_db[table_db['bandwidth'] != 54]
    widths = db_l["width_ms"].values
    print(np.median(widths))
    print(np.min(widths))
    print(np.max(widths))

    # db_l = table_db[table_db['bandwidth'] == 54]
    # widths = db_l["width_ms"].values
    # print(np.median(widths))
    # print(np.min(widths))
    
    return
#max_values()

In [11]:
def calc_time_delay(freq, dm, dmconst):
    """
    Input:
        freq (MHz), float
        dm (pc/cc), float
        dmconst (GHz^2 cm^3 pc^-1 ms), float
    Returns:
        The time difference between infinite frequency and freq
        in SECONDS"""
    return dmconst * 10**6. * dm * freq**-2. / 1000.

In [12]:
from astropy.coordinates import EarthLocation
from astropy import coordinates as coord
import astropy.units as u
from astropy.time import Time

def loc2bary(mjd, dm_const=1/0.241):

    #Via Dante
    ref_freq = 1738

    #DM used for all bursts in NRT analysis
    dm = 464.86

    #Coordinates of NRT
    source = coord.SkyCoord('19:49:29.21', '-25:12:49.40', unit=(u.hourangle, u.deg))

    #NRT via D. Hewitt
    station = EarthLocation.from_geocentric(x=4324165.81 * u.m,
                                            y=165927.11 * u.m,
                                            z=4670132.83 * u.m)

    #Define the local MJD and the location
    localTOA = Time(mjd, format='mjd', scale='utc', location=station)    
    delay_to_inf = calc_time_delay(ref_freq, dm, dm_const) * u.s  # in seconds  
    delay_to_bary = localTOA.light_travel_time(source, 'barycentric')  # in TDB seconds, to be added
    baryTOA = localTOA - delay_to_inf + delay_to_bary
    
    return baryTOA.tdb.value

#loc2bary(mjd=[60511.987776,60511.987776], dm_const=1/0.241)

In [21]:
def nancay_rmkt(csv_file, mjd_limit=True):
    """
    Imports, processes, and merges fluence and rotation measure (RM) data 
    for the Nançay Radio Telescope (NRT) burst sample.

    This function:
    - Loads a CSV file with burst properties.
    - Renames and standardizes relevant columns.
    - Converts local MJDs to barycentric TDB using `loc2bary`.
    - Computes fluence uncertainties as 20% of the measured fluence.
    - Rounds and cleans various fields.
    - Loads a separate CSV with RM measurements and merges it with the main DataFrame 
      based on rounded MJD values.
    - Reorders and filters columns.
    - Saves the resulting database to a new CSV file.

    Parameters
    ----------
    csv_file : str
        Path to the CSV file containing the NRT burst data.
    mjd_limit : bool, optional
        Currently unused; placeholder for future functionality to filter on MJD.

    Returns
    -------
    None
        The function displays the final DataFrame and saves it to a CSV file.
    """
    
    nancay_df = pd.read_csv(csv_file, on_bad_lines='skip', header=0, sep=';')
    #print(nancay_df.columns)
    
    nancay_df.rename(columns={'MJD_at_peak': 'MJD_local'}, inplace=True)
    nancay_df.rename(columns={'burst_name': 'burst_name_internal'}, inplace=True)
    nancay_df.rename(columns={'event_duration_ms': 'width_ms'}, inplace=True)
    
    mjd_local_vals = nancay_df["MJD_local"].values
    mjd_bary_vals = loc2bary(mjd_local_vals)

    nancay_df["MJD_bary_tdb"] = mjd_bary_vals

    # Round the 'fluence_Jyms' column to two decimal places
    nancay_df['fluence_Jyms'] = nancay_df['fluence_Jyms'].round(3)
    nancay_df['width_ms'] = nancay_df['fluence_Jyms'].round(2)
    nancay_df['spectral_extent_MHz'] = nancay_df['spectral_extent_MHz'].astype(int)
    
    # Create a new column 'fluence_Jyms_error' as 20% of 'fluence_Jyms'
    nancay_df['fluence_Jyms_error'] = nancay_df['fluence_Jyms'] * 0.20
    
    # Select only the desired columns
    columns_to_keep = ['MJD_bary_tdb', 'peak_flux', 'fluence_Jyms', 'fluence_Jyms_error', 'width_ms', 'spectral_extent_MHz','burst_name_internal', 'MJD_local']
    df_filtered = nancay_df[columns_to_keep].sort_values(by='MJD_bary_tdb', ascending=True).reset_index(drop=True)
    
    ###
    #Adding the RM information in to the db 
    ###

    ##Loading in the csv file
    rm_df = pd.read_csv("NRT_RM_values_and_errors.csv", header=0, index_col=0)

    #Now we introduce the mjd_rounded variable, so we can easily merge the dfs with each other
    df_filtered['mjd_rounded'] = df_filtered['MJD_local'].astype(float).round(6)
    rm_df['mjd_rounded'] = rm_df['MJD'].astype(float).round(6)

    #Using left, we keep all lines and Nans are introduced
    merged = pd.merge(df_filtered, rm_df, on='mjd_rounded', how='left')

    ##Now we drop some columns
    df_nrt = merged.drop(columns=['MJD','mjd_rounded', 'RM_pcm60499', 'RM_pcm60598'])

    # Create custom index column with prefix 'B' and numbers from B001 to B206
    df_nrt['index'] = ['B' + str(i).zfill(3) for i in range(1, 207)]
    
    # Move the index column to the front (optional)
    df_nrt = df_nrt[['index'] + [col for col in df_nrt.columns if col != 'index']]

    desired_order = [
                'index','MJD_bary_tdb', 'peak_flux', 'fluence_Jyms', 'fluence_Jyms_error',
                'width_ms', 'spectral_extent_MHz',
                'fit_RM_pcm60499', 'fit_RM_err_pcm60499',
                'fit_RM_pcm60598', 'fit_RM_err_pcm60598',
                'MJD_local', 'burst_name_internal'
                    ]
    df_nrt = df_nrt[desired_order]

    display(df_nrt)
    #Saving the db
    name_csv = "NRT_burst_db_bary_pol_v2.csv"
    #df_nrt.to_csv(name_csv, header=True, index=True, na_rep='NA')
    
    return df_nrt

df_nrt = nancay_rmkt(csv_file="/home/omar/git/frb20240619d-single-dish-analysis/dbs/nrt/nrt_burst_db_v2.csv",\
                                 mjd_limit=False)

,index,MJD_bary_tdb,peak_flux,fluence_Jyms,fluence_Jyms_error,width_ms,spectral_extent_MHz,fit_RM_pcm60499,fit_RM_err_pcm60499,fit_RM_pcm60598,fit_RM_err_pcm60598,MJD_local,burst_name_internal
0,B001,60497.026715,0.616850,0.755,0.1510,0.76,144,NaN,NaN,NaN,NaN,60497.020158,burst_60497.0201579919s1p1t46.6505_500ms
1,B002,60497.026983,1.633723,3.883,0.7766,3.88,188,-180.20,31.85,-187.41,28.15,60497.020426,burst_60497.020426341434s1p1t69.8359_500ms
2,B003,60497.027644,0.815267,2.363,0.4726,2.36,432,NaN,NaN,NaN,NaN,60497.021087,burst_60497.02108689699s1p2t6.9079_500ms
3,B004,60497.031234,0.623390,0.983,0.1966,0.98,428,NaN,NaN,NaN,NaN,60497.024677,burst_60497.02467708565s1p4t77.1002_500ms
4,B005,60497.033447,0.728133,1.239,0.2478,1.24,188,NaN,NaN,NaN,NaN,60497.026890,burst_60497.02688972801s1p6t28.2725_500ms
...,...,...,...,...,...,...,...,...,...,...,...,...,...
201,B202,60537.918644,0.798502,3.366,0.6732,3.37,184,NaN,NaN,NaN,NaN,60537.912721,burst_60537.912720859305s3p3t15.3801_500ms
202,B203,60537.922310,1.195546,0.762,0.1524,0.76,184,NaN,NaN,NaN,NaN,60537.916387,burst_60537.91638674126s3p5t92.1123_500ms
203,B204,60537.932108,0.716661,1.402,0.2804,1.40,112,NaN,NaN,NaN,NaN,60537.926185,burst_60537.926184708194s5p1t76.9545_500ms
204,B205,60537.935503,3.054093,1.805,0.3610,1.80,232,-254.94,43.14,-272.43,30.64,60537.929581,burst_60537.92958073482s5p4t10.3712_500ms


In [22]:
df_nrt.sort_values(by=["fluence_Jyms"])

,index,MJD_bary_tdb,peak_flux,fluence_Jyms,fluence_Jyms_error,width_ms,spectral_extent_MHz,fit_RM_pcm60499,fit_RM_err_pcm60499,fit_RM_pcm60598,fit_RM_err_pcm60598,MJD_local,burst_name_internal
80,B081,60516.980134,0.711896,0.342,0.0684,0.34,144,NaN,NaN,NaN,NaN,60516.973548,burst_60516.97354816487s3p5t76.8593_500ms
108,B109,60516.995769,0.725488,0.424,0.0848,0.42,224,NaN,NaN,NaN,NaN,60516.989183,burst_60516.98918321051s5p5t86.0251_500ms
177,B178,60520.980965,0.638929,0.453,0.0906,0.45,204,NaN,NaN,NaN,NaN,60520.974452,burst_60520.97445168338s3p6t31.9233_500ms
89,B090,60516.984954,0.723607,0.454,0.0908,0.45,252,NaN,NaN,NaN,NaN,60516.978368,burst_60516.97836831269s4p3t62.469_500ms
27,B028,60511.997082,0.809274,0.463,0.0926,0.46,112,NaN,NaN,NaN,NaN,60511.990442,burst_60511.990442212315s3p5t109.505_500ms
...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,B093,60516.988155,0.944511,9.483,1.8966,9.48,316,NaN,NaN,NaN,NaN,60516.981570,burst_60516.98156901176s4p5t99.0094_500ms
101,B102,60516.993132,1.398201,10.424,2.0848,10.42,144,NaN,NaN,NaN,NaN,60516.986547,burst_60516.98654671515s5p3t98.2319_500ms
37,B038,60512.008530,1.265088,11.697,2.3394,11.70,488,-239.41,25.45,-238.55,25.64,60512.001891,burst_60512.00189087138s5p2t116.967_500ms
95,B096,60516.988692,2.078187,18.950,3.7900,18.95,376,-243.03,18.08,-241.09,20.15,60516.982106,burst_60516.98210609625s4p6t25.4135_500ms
